In [1]:
!pip install papermill


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import pandas as pd

In [3]:
# ==========================================
#  ZONE DE CONTRÔLE (À MODIFIER À LA MAIN)
# ==========================================

ma_region = 'Bourgogne_Franche_Comte'
region_code = 'BFC'
mes_departements = ['21','25','39','58','70','71','89']

In [4]:
import papermill as pm

def run_pipeline_for_region(nom_region, region_code, departements):
    print(f"Démarrage de la pipeline pour la région : {nom_region}")
    print(f"Départements à traiter : {departements}")
    
    nom_fichier_region = f'dpe_{region_code}.csv'

    # 1. Téléchargement pour chaque département
    if not os.path.exists(nom_fichier_region):
        for dep in departements:
            print(f"Téléchargement des données pour le département {dep}...")
            pm.execute_notebook(
                'DL_dep_DPE.ipynb',
                f'DL_dep_DPE_output_{dep}.ipynb', # Nom du notebook de sortie avec les logs
                parameters=dict(Dep=dep)
            )

        # ---  Affichage de la taille du dataset ---
            # On cherche le nom du fichier CSV sauvegardé par votre notebook
            nom_fichier = f'Dpe_dep_{dep}.csv'
            
            if os.path.exists(nom_fichier):
                # Astuce : on ne lit qu'une seule colonne (usecols=[0]) pour que ça soit instantané 
                # et ne pas saturer la mémoire de l'ordinateur
                taille_dataset = len(pd.read_csv(nom_fichier, usecols=[0]))
                print(f"   ---> ✅ Dataset {dep} prêt : {taille_dataset} logements récupérés.\n")
            else:
                print(f"   ---> ⚠️ Attention, le fichier {nom_fichier} n'a pas été trouvé après exécution.\n")
        
        # 2. Agrégation des données en une région
        print(f"Fusion des départements pour la région {nom_region}...")
        pm.execute_notebook(
            'get_data_region.ipynb',
            f'get_data_region_output_{nom_region}.ipynb',
            parameters=dict(region_code=region_code, departements=departements)
        )

    # ---  Affichage de la taille du dataset régional ---
    
    if os.path.exists(nom_fichier_region):
        taille_dataset_region = len(pd.read_csv(nom_fichier_region, usecols=[0]))
        print(f"   ---> 🌍 Dataset régional ({region_code}) prêt : {taille_dataset_region} logements au total réunis !\n")
    else:
        print(f"   ---> Attention, le fichier {nom_fichier_region} n'a pas été trouvé après la fusion.\n")

    # 3. Création du modèle électrique
    print(f"Création et entraînement du modèle électrique pour {nom_region}...")
    pm.execute_notebook(
        'creation_model_elec.ipynb',
        f'creation_model_elec_output_{nom_region}.ipynb',
        parameters=dict(nom_region=nom_region,region_code=region_code)
    )

     # 4. Création du modèle DPE
    print(f"Création et entraînement du modèle DPE pour {nom_region}...")
    pm.execute_notebook(
        'creation_model_dpe.ipynb',
        f'creation_model_dpe_output_{nom_region}.ipynb',
        parameters=dict(nom_region=nom_region,region_code=region_code)
    )
    
    print(f"✅ Pipeline terminée avec succès pour {nom_region} !")


# Lancement
run_pipeline_for_region(ma_region, region_code,mes_departements)

Démarrage de la pipeline pour la région : Bourgogne_Franche_Comte
Départements à traiter : ['21', '25', '39', '58', '70', '71', '89']


c:\USERS\LILYM\DOCUMENTS\CAPSTONE\PROJET-DPE\ANALYSE\.VENV\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


   ---> 🌍 Dataset régional (BFC) prêt : 496650 logements au total réunis !

Création et entraînement du modèle électrique pour Bourgogne_Franche_Comte...


Executing: 100%|██████████| 14/14 [01:00<00:00,  4.33s/cell]


Création et entraînement du modèle DPE pour Bourgogne_Franche_Comte...


Executing: 100%|██████████| 10/10 [00:49<00:00,  4.95s/cell]

✅ Pipeline terminée avec succès pour Bourgogne_Franche_Comte !


In [ ]:
import os # <-- Indispensable pour supprimer des fichiers

def delete_temp_files(nom_region, region_code, departements):
    
    #  ETAPE DE NETTOYAGE A LA TOUTE FIN :
    print("Nettoyage des fichiers temporaires...")
    
    # On supprime les fichiers des départements
    for dep in departements :
        if os.path.exists( f'DL_dep_DPE_output_{dep}.ipynb'):
            os.remove( f'DL_dep_DPE_output_{dep}.ipynb')

    # On supprime le fichier de la région
    if os.path.exists(f'get_data_region_output_{nom_region}.ipynb'):
        os.remove(f'get_data_region_output_{nom_region}.ipynb')
        
    # On supprime le fichier du modèle
    if os.path.exists(f'creation_model_elec_output_{nom_region}.ipynb'):
        os.remove(f'creation_model_elec_output_{nom_region}.ipynb')

    if os.path.exists(f'creation_model_dpe_output_{nom_region}.ipynb'):
        os.remove(f'creation_model_dpe_output_{nom_region}.ipynb')

        
    print("Pipeline terminée !")

In [6]:
delete_temp_files(ma_region, region_code, mes_departements)

Nettoyage des fichiers temporaires...
Pipeline terminée et dossier tout propre !


In [ ]:
def delete_dpe_files(nom_region, region_code, departements):
    
    #  ETAPE DE NETTOYAGE A LA TOUTE FIN :
    print("Nettoyage des fichiers temporaires...")
    
    # On supprime les fichiers des départements
    for dep in departements :
        if os.path.exists( f'Dpe_dep_{dep}.ipynb'):
            os.remove( f'Dpe_dep_{dep}.ipynb')

    # On supprime le fichier de la région
    if os.path.exists(f'dpe_{nom_region}.ipynb'):
        os.remove(f'dpe_{nom_region}.ipynb')
                
    print("Pipeline terminée !")